## Test 1 — B=1 regression

Single PIL image + 1-D siglip. Confirms return type is `Image.Image` and
that two back-to-back calls with the same seed produce pixel-exact outputs
(VAE encode uses `.mode()` so encoding is deterministic).

In [1]:
import sys
import importlib
import torch
import numpy as np
from PIL import Image

sys.path.insert(0, '.')

# Force reload of generation module to pick up any on-disk changes.
for mod in list(sys.modules.keys()):
    if mod.startswith('generation'):
        del sys.modules[mod]

from config_const import (
    SEED, HVM_STIM_DIR, HVM_N_VAR, HVM_CATEGORIES,
    HVM_SIGLIP_EMBEDDINGS_PATH,
)
from data_utils.hvm_loader import _category_stratified_split
from generation.flux_instantx import load_pipeline, generate_img2img, encode_image
from generation.aperture import load_hvm_packed_aperture_mask, build_object_region_mask
from get_device import get_device

# Sanity-check that encode_image uses .mode() not .sample()
import inspect
src = inspect.getsource(encode_image)
assert 'mode()' in src and 'sample()' not in src, \
    'encode_image still uses .sample() — check flux_instantx.py'
print('encode_image uses .mode() ✓')

DEVICE     = get_device()
IMAGE_SIZE = 512
STRENGTH   = 0.65
STEPS      = 20
SEED_GEN   = 7

pipe, image_proj = load_pipeline(device=DEVICE, default_scale=1.0)
aperture = load_hvm_packed_aperture_mask(image_size=IMAGE_SIZE, device='cpu', dtype=torch.bfloat16)

_, _, test_idx = _category_stratified_split(SEED)
test_idx = test_idx[:4]
siglip_gt = torch.load(HVM_SIGLIP_EMBEDDINGS_PATH, weights_only=True)

def load_stim(i):
    cat = HVM_CATEGORIES[i // HVM_N_VAR]
    var = i % HVM_N_VAR
    return Image.open(HVM_STIM_DIR / cat / f'{var:02d}.png').convert('RGB').resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)

originals  = [load_stim(int(i)) for i in test_idx]
embeddings = siglip_gt[test_idx]   # (4, 1152)
print(f'loaded {len(originals)} stimuli, embeddings {tuple(embeddings.shape)}')

encode_image uses .mode() ✓


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

loaded 4 stimuli, embeddings (4, 1152)


In [2]:
def shared_kwargs(seed=SEED_GEN):
    return dict(
        strength=STRENGTH, prompt='',
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        num_inference_steps=STEPS, guidance_scale=3.5,
        ip_adapter_scale=1.0, aperture_mask=aperture,
        seed=seed, show_progress=False,
    )

def imgs_equal(a, b):
    return np.array_equal(np.array(a), np.array(b))

def max_abs_diff(a, b):
    return int(np.abs(np.array(a).astype(int) - np.array(b).astype(int)).max())

def assert_close(a, b, label, tol=20):
    d = max_abs_diff(a, b)
    assert d <= tol, f'{label}: max pixel diff {d} exceeds tolerance {tol}'
    return d

In [3]:
out_a = generate_img2img(pipe, image_proj, originals[0], embeddings[0], **shared_kwargs())
out_b = generate_img2img(pipe, image_proj, originals[0], embeddings[0], **shared_kwargs())
assert isinstance(out_a, Image.Image), f'expected Image.Image, got {type(out_a)}'
assert imgs_equal(out_a, out_b), f'B=1 not deterministic, max diff={max_abs_diff(out_a, out_b)}'
print('Test 1 PASSED: B=1 returns Image.Image and is deterministic')

Test 1 PASSED: B=1 returns Image.Image and is deterministic


## Test 2 — B=4 identical inputs

Same image and embedding in all four slots. The B=1 reference and the B=4
batch are generated consecutively. bf16 accumulation order differs between
batch sizes, so outputs are numerically close but not necessarily bit-exact;
we assert max pixel diff ≤ 10 (uint8 scale).

In [12]:
ref = generate_img2img(pipe, image_proj, originals[0], embeddings[0], **shared_kwargs())

emb_batch  = embeddings[0:1].expand(4, -1).contiguous()
imgs_batch = [originals[0]] * 4
out_batch  = generate_img2img(pipe, image_proj, imgs_batch, emb_batch, **shared_kwargs())

assert isinstance(out_batch, list) and len(out_batch) == 4, \
    f'expected list[Image.Image] len 4, got {type(out_batch)}'

# Slots must be pixel-exact with each other (same input → same noise → same output).
for k in range(1, 4):
    assert imgs_equal(out_batch[k], out_batch[0]), \
        f'slot {k} differs from slot 0, max_diff={max_abs_diff(out_batch[k], out_batch[0])}'

# Each slot should be close to the B=1 ref (bf16 accumulation-order difference).
diffs = [assert_close(out_k, ref, f'batch[{k}]') for k, out_k in enumerate(out_batch)]
print(f'Test 2 PASSED: slots identical to each other, close to B=1 ref (max diffs: {diffs})')

Test 2 PASSED: slots identical to each other, close to B=1 ref (max diffs: [12, 12, 12, 12])


## Test 3 — B=4 distinct inputs

Four different stimuli. B=1 references run immediately before the batched
call. Asserts max pixel diff ≤ 10 per sample.

In [13]:
b1_results = [
    generate_img2img(pipe, image_proj, originals[k], embeddings[k], **shared_kwargs())
    for k in range(4)
]
batched_results = generate_img2img(
    pipe, image_proj, originals[:4], embeddings[:4], **shared_kwargs())

diffs = [
    assert_close(batched_results[k], b1_results[k], f'sample {k}')
    for k in range(4)
]
print(f'Test 3 PASSED: B=4 distinct inputs close to B=1 (max diffs: {diffs})')

Test 3 PASSED: B=4 distinct inputs close to B=1 (max diffs: [12, 12, 6, 4])


## Test 4 — B=4 with per-sample `object_mask`

Four masks with different bbox radii (different `n_bbox` token counts per
sample). Exercises the per-sample loop in `IPAFluxAttnProcessor`.

In [14]:
cx, cy = IMAGE_SIZE / 2, IMAGE_SIZE / 2
radii   = [64, 96, 128, 160]   # different n_bbox per sample
masks_b1 = [
    build_object_region_mask(IMAGE_SIZE, cx, cy, r, device='cpu', dtype=torch.bfloat16)
    for r in radii
]  # each (1, 1024, 1)
obj_emb = embeddings[0]

b1_mask_results = [
    generate_img2img(
        pipe, image_proj, originals[k], embeddings[k],
        object_siglip_embedding=obj_emb,
        object_mask=masks_b1[k],
        object_ip_scale=0.5,
        **shared_kwargs(),
    )
    for k in range(4)
]

masks_batched   = torch.cat(masks_b1, dim=0)             # (4, 1024, 1)
obj_emb_batch   = obj_emb.unsqueeze(0).expand(4, -1)     # (4, 1152)
batched_mask_results = generate_img2img(
    pipe, image_proj, originals[:4], embeddings[:4],
    object_siglip_embedding=obj_emb_batch,
    object_mask=masks_batched,
    object_ip_scale=0.5,
    **shared_kwargs(),
)

diffs = [
    assert_close(batched_mask_results[k], b1_mask_results[k], f'object_mask sample {k}')
    for k in range(4)
]
print(f'Test 4 PASSED: per-sample object_mask loop close to B=1 (max diffs: {diffs})')

Test 4 PASSED: per-sample object_mask loop close to B=1 (max diffs: [20, 11, 8, 13])


## Test 5 — Guard clauses

`use_rf_inversion=True` with B > 1 must raise `NotImplementedError`.

In [15]:
try:
    generate_img2img(
        pipe, image_proj, originals[:2], embeddings[:2],
        use_rf_inversion=True, **shared_kwargs())
    assert False, 'expected NotImplementedError'
except NotImplementedError:
    pass

print('Test 5 PASSED: use_rf_inversion=True with B > 1 raises NotImplementedError')

Test 5 PASSED: use_rf_inversion=True with B > 1 raises NotImplementedError


## Timing and max batch size

Time B=1 through increasing batch sizes and find the OOM limit.
Uses the first N test stimuli cycled to fill each batch.

In [17]:
import time

# Load enough stimuli to fill up to B=8 with distinct images.
all_idx   = _category_stratified_split(SEED)[2]   # test split
all_stims = [load_stim(int(i)) for i in all_idx[:8]]
all_embs  = siglip_gt[all_idx[:8]]

results = []   # (B, wall_s, peak_gb)
max_ok  = 0

for B in [1, 2, 3, 4, 5, 6, 7, 8]:
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    try:
        imgs = all_stims[:B]
        embs = all_embs[:B]
        t0 = time.perf_counter()
        generate_img2img(pipe, image_proj, imgs, embs, **shared_kwargs())
        wall = time.perf_counter() - t0
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        results.append((B, wall, peak_gb))
        max_ok = B
        print(f'B={B}: {wall:.1f}s  peak={peak_gb:.1f} GB')
    except torch.cuda.OutOfMemoryError:
        print(f'B={B}: OOM')
        break

print(f'\nMax batch size: {max_ok}')
if len(results) > 1:
    b1_time = results[0][1]
    print(f'Throughput vs B=1 (images/s):')
    for B, wall, gb in results:
        print(f'  B={B}: {B/wall:.2f} img/s  ({B/wall / (1/b1_time):.1f}x speedup)')

B=1: 5.0s  peak=39.7 GB
B=2: 9.3s  peak=39.7 GB
B=3: 13.6s  peak=39.7 GB
B=4: 17.8s  peak=39.8 GB
B=5: 22.0s  peak=39.9 GB
B=6: 26.3s  peak=40.1 GB
B=7: 29.7s  peak=40.3 GB
B=8: 34.1s  peak=40.4 GB

Max batch size: 8
Throughput vs B=1 (images/s):
  B=1: 0.20 img/s  (1.0x speedup)
  B=2: 0.21 img/s  (1.1x speedup)
  B=3: 0.22 img/s  (1.1x speedup)
  B=4: 0.23 img/s  (1.1x speedup)
  B=5: 0.23 img/s  (1.1x speedup)
  B=6: 0.23 img/s  (1.1x speedup)
  B=7: 0.24 img/s  (1.2x speedup)
  B=8: 0.23 img/s  (1.2x speedup)


In [16]:
print('All tests passed.')

All tests passed.
